# Module 2 POC Training -- FIRE + LongDRScreening + Tianjin

Runs `train_module2_poc.py` against all three Module 2 data sources at once: FIRE and
LongDRScreening (placeholder Stage-2 conditioning unless a Module 1 cache says otherwise,
same as notebook 04) plus Tianjin (real clinical DR grade conditioning, from notebook 05's
cache/xlsx read).

**Run this after notebook 04** (FIRE/LongDR Module 1 caches) **and notebook 05** (Tianjin
Module 1 cache) **have both finished** -- this notebook only reads their cache outputs from
Drive, it doesn't rebuild them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'main'  # <-- change if the code you need (e.g. this Tianjin work) isn't merged yet
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache')
for _name in ['module1_outputs_fire.pt', 'module1_outputs_longdr.pt', 'module1_outputs_tianjin.pt']:
    _p = os.path.join(CACHE_DIR, _name)
    assert os.path.isfile(_p), (
        f"Not found: {_p} -- run notebook 04 (FIRE/LongDR) and notebook 05 (Tianjin) first."
    )
    print('✓', _p)

In [ ]:
# Unzip the three raw datasets locally (train_module2_poc.py's loaders read images directly
# at training time, not just at cache-build time). Same unzip conventions as notebooks 01/04/05.
import glob, shutil

os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FIRE_dataset.zip"
!unzip -q -n "$DRIVE_DATA_DIR/LongDRScreening_20150209.zip" -d LongDRScreening_20150209
!unzip -q -n "$DRIVE_DATA_DIR/retinal-dr-longitudinal.zip" -d _tianjin_extract_raw

_manifest_candidates = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _manifest_candidates, 'No corrected_manifest.csv found -- see notebook 05 for troubleshooting.'
_tianjin_root = os.path.dirname(_manifest_candidates[0])
TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if _tianjin_root != TIANJIN_DIR and not os.path.exists(TIANJIN_DIR):
    os.symlink(_tianjin_root, TIANJIN_DIR)

print('FIRE Images present   :', os.path.isdir('/content/data/FIRE_dataset/FIRE/Images'))
print('LongDR norm present   :', os.path.isdir('/content/data/LongDRScreening_20150209/FundusImagesNormalized'))
print('Tianjin manifest found:', os.path.isfile(os.path.join(TIANJIN_DIR, 'corrected_manifest.csv')))

In [ ]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}

REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"

%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch torchmetrics torch-fidelity

In [ ]:
# --fire-dir / --longdr-dir / --tianjin-dir MUST be passed explicitly: train_module2_poc.py's
# own argparse defaults resolve relative to the cloned repo's working directory, not where the
# data actually lives on the Colab VM (see combined_dataset.py / train_module2_poc.py docstrings).
%cd {REPO_CODE_DIR}
!python train_module2_poc.py \
    --fire-dir /content/data/FIRE_dataset \
    --longdr-dir /content/data/LongDRScreening_20150209 \
    --tianjin-dir "{TIANJIN_DIR}" \
    --fire-module1-cache /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_fire.pt \
    --longdr-module1-cache /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_longdr.pt \
    --tianjin-module1-cache /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_tianjin.pt \
    --num-epochs 5

In [ ]:
from IPython.display import Image as IPImage, display
import glob, json, os

samples = sorted(glob.glob(f'{REPO_CODE_DIR}/training_samples_poc/epoch_*.jpg'))
if samples:
    display(IPImage(samples[-1]))
else:
    print('No training_samples_poc/epoch_*.jpg found -- the training cell above did not finish. Read its output.')

rp = f'{REPO_CODE_DIR}/DRForestGAN-v2/stargan/models_poc/poc_results.json'
print(json.dumps(json.load(open(rp)), indent=2) if os.path.isfile(rp) else f'{rp} not found')

## Honest caveats for the presentation

- Same POC-scale caveats as notebook 04 (5 epochs, manuscript-matched Adam settings, EX+MA-only
  segmentation conditioning, FID/PSNR/SSIM from a tiny short run).
- Tianjin adds the pipeline's first *real* clinical grade conditioning source -- but the
  ETDRS(1-5)->ICDR(0-4) mapping is a documented, unverified assumption (see
  `tianjin_dataset.py` and `docs/DATASETS.md`); don't present Tianjin-conditioned results as a
  validated grading-accuracy claim without adviser/clinical sign-off.
- FIRE and LongDR still fall back to placeholder Stage-2 conditioning wherever no Module 1
  cache entry exists for a given image -- this run does not change that.
- Module 2's AdaIN stage-conditioning + autoregressive multi-stage cascade generation (the
  manuscript's claimed core original contribution) is still not implemented -- this run uses
  the existing single-step conditional GAN in `DRForestGAN-v2/`. See `docs/ROADMAP.md`.